In [ ]:
# ── Cell 1: Imports & Parameters ─────────────────────────────────────────────
# Notebook 17e: RF + MobileNetV2 Features — Real Only (path_mnist)
# Uses pre-extracted MobileNetV2 features from notebook 13
# Hyperparameters fixed from notebook 17a HP search

import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

SEED = 42
import numpy as np
np.random.seed(SEED)

CLASS_NAMES = ['Adipose', 'Background', 'Debris', 'Lymphocytes', 'Mucus', 'Smooth Muscle', 'Normal Colon Mucosa', 'Cancer-Assoc. Stroma', 'Colorectal Adenocarcinoma']

# Fixed hyperparameters from notebook 17a HP search
CRITERION    = 'gini'
MAX_DEPTH    = 100
N_ESTIMATORS = 100

PROJECT_DIR       = '/content/drive/MyDrive/thesis_project'
DDIM_FEATURES_DIR = os.path.join(PROJECT_DIR, 'data', 'extracted_features', 'seed_42', 'ddim')
REAL_FEATURES_DIR = os.path.join(PROJECT_DIR, 'data', 'extracted_features', 'seed_42', 'real')
SYNTH_NPZ         = os.path.join(PROJECT_DIR, 'data', 'synthetic', 'seed_42', 'ddim', 'path_mnist/path_mnist_synthetic.npz')
REAL_NPZ          = os.path.join(PROJECT_DIR, 'data', 'processed', 'seed_42', 'path_mnist.npz')
RESULTS_DIR       = os.path.join(PROJECT_DIR, 'results', 'rf_features', 'path_mnist', 'real_only')
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 2: Load Pre-extracted Features & Labels ──────────────────────────────

F_train_real = np.load(os.path.join(REAL_FEATURES_DIR, 'path_mnist_x_train_features.npy'))
F_val_real   = np.load(os.path.join(REAL_FEATURES_DIR, 'path_mnist_x_val_features.npy'))
F_train      = np.concatenate([F_train_real, F_val_real], axis=0)

real_data = np.load(REAL_NPZ)
y_train   = np.concatenate([real_data['y_train'], real_data['y_val']], axis=0)

F_test = np.load(os.path.join(REAL_FEATURES_DIR, 'path_mnist_x_test_features.npy'))
y_test = real_data['y_test']

print(f'F_train: {F_train.shape}, y_train: {y_train.shape}')
print(f'F_test:  {F_test.shape},  y_test:  {y_test.shape}')

In [ ]:
# ── Cell 3: Train & Evaluate ──────────────────────────────────────────────────

clf = RandomForestClassifier(
    criterion=CRITERION,
    max_depth=MAX_DEPTH,
    n_estimators=N_ESTIMATORS,
    random_state=SEED,
    n_jobs=-1
)
clf.fit(F_train, y_train)

y_pred   = clf.predict(F_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {test_acc:.4f}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

report_dict = classification_report(
    y_test, y_pred, target_names=CLASS_NAMES, digits=4, output_dict=True
)
pd.DataFrame(report_dict).transpose().to_csv(
    os.path.join(RESULTS_DIR, 'classification_report.csv')
)
pd.DataFrame([{
    'condition': 'real_only',
    'dataset':   'path_mnist',
    'test_acc':  round(test_acc, 4),
}]).to_csv(os.path.join(RESULTS_DIR, 'summary.csv'), index=False)
print(f'\nResults saved to: {RESULTS_DIR}')